# Clasificación y evaluación de modelos con `scikit-learn` — Sesión 2  
## Bloque 3: clasificación supervisada  
## Bloque 4: evaluación, validación e hiperparámetros

**Objetivo del cuaderno**  
Este notebook está pensado para alumnado que ya ha visto una primera sesión de introducción al flujo general de Machine Learning y a la regresión, y que ahora necesita una entrada muy guiada al mundo de la **clasificación**.

La idea no es solo "entrenar un clasificador", sino entender con mucha calma:

- qué significa clasificar,
- en qué se parece y en qué se diferencia respecto a la regresión,
- cómo se organiza un proyecto de clasificación con `scikit-learn`,
- qué papel juegan los **transformadores** y los **predictores**,
- por qué usamos `train_test_split`, validación cruzada y `GridSearchCV`,
- por qué **accuracy** puede quedarse corta,
- y cómo leer métricas como **precision**, **recall**, **F1**, **ROC-AUC** y la **matriz de confusión**.

Trabajaremos con el dataset público **Breast Cancer Wisconsin**, incluido en `scikit-learn`.  
Lo usaremos como recurso docente para aprender a clasificar entre dos clases:

- `malignant`
- `benign`

> Importante: el contexto del dataset es médico, pero aquí lo usamos **solo con fines educativos**.  
> No estamos construyendo una herramienta clínica, sino aprendiendo conceptos y herramientas de Machine Learning.

## Hoja de ruta

### Parte A — Bloque 3
1. Qué es una clasificación.
2. Clasificación binaria frente a multiclase.
3. Clase predicha frente a probabilidad predicha.
4. Flujo general de trabajo en clasificación.
5. Recordatorio de la API mental de `scikit-learn`.
6. Transformadores vs predictores.
7. Dataset de trabajo y análisis inicial.
8. `train_test_split` con estratificación.
9. Baseline y primer modelo serio.

### Parte B — Bloque 4
10. Métricas de clasificación.
11. Matriz de confusión e informe de clasificación.
12. `predict` frente a `predict_proba`.
13. Umbral de decisión y compromiso precision/recall.
14. Validación cruzada.
15. Comparación de varios modelos.
16. `GridSearchCV` y ajuste de hiperparámetros.
17. Evaluación final honesta sobre test.
18. Conclusiones y errores frecuentes.

# Parte A. Clasificación supervisada

## 1. ¿Qué es una clasificación?

En un problema de **clasificación**, la variable objetivo no es un número continuo, sino una **etiqueta** o **clase**.

Ejemplos típicos:

- spam / no spam,
- fraude / no fraude,
- abandono / no abandono,
- perro / gato / pájaro,
- benign / malignant.

La lógica general sigue siendo la del aprendizaje supervisado:

- tenemos datos de entrada `X`,
- tenemos una salida conocida `y`,
- y entrenamos un modelo para aprender una relación entre ambos.

La diferencia con la regresión es que en clasificación no queremos predecir una cantidad, sino **decidir entre categorías**.

## 2. Clasificación binaria y clasificación multiclase

### Clasificación binaria
Hay solo **dos clases** posibles.

Ejemplos:
- sí / no,
- aprueba / suspende,
- benign / malignant.

### Clasificación multiclase
Hay **más de dos clases** posibles.

Ejemplos:
- especie de flor,
- tipo de vino,
- dígito escrito.

En este notebook trabajaremos con un problema **binario**, porque es una forma muy buena de introducir:

- probabilidades,
- umbral de decisión,
- precision,
- recall,
- F1,
- ROC-AUC,
- y tipos de error.

## 3. Clase predicha frente a probabilidad predicha

Esta idea es muy importante.

Un clasificador puede devolvernos:

### a) La clase final
Por ejemplo:
- `benign`
- `malignant`

Esto suele obtenerse con `predict`.

### b) Una probabilidad o score
Por ejemplo:
- probabilidad de `malignant` = 0.82
- probabilidad de `benign` = 0.18

Esto suele obtenerse con `predict_proba` cuando el modelo lo permite.

¿Por qué importa?

Porque en muchos contextos no basta con saber la clase final.  
Nos interesa saber también **con qué seguridad** parece estar decidiendo el modelo.

## 4. Flujo típico de trabajo en clasificación

El flujo general se parece mucho al de regresión:

1. Entender el problema.
2. Cargar y comprender los datos.
3. Definir `X` e `y`.
4. Separar train y test.
5. Preparar los datos si hace falta.
6. Crear un baseline.
7. Entrenar uno o varios modelos.
8. Evaluar con métricas adecuadas.
9. Comparar modelos y ajustar hiperparámetros.
10. Reservar el test para la evaluación final honesta.

La gran diferencia está en cómo medimos el rendimiento y cómo interpretamos los errores.

## 5. Recordatorio: la API mental de `scikit-learn`

Merece la pena insistir en esto porque `scikit-learn` está diseñado con una interfaz muy coherente.

### Métodos clave

#### `fit`
Ajusta el objeto con datos de entrenamiento.

- En un predictor, aprende una relación entre `X` e `y`.
- En un transformador, aprende cómo transformar los datos.

#### `transform`
Convierte unos datos en otra representación.

#### `fit_transform`
Hace `fit` y luego `transform`.

#### `predict`
Devuelve etiquetas predichas.

#### `predict_proba`
Devuelve probabilidades por clase.

#### `score`
Devuelve una métrica por defecto del objeto.  
En clasificación suele ser accuracy, pero normalmente conviene indicar las métricas de forma explícita.

## 6. Transformadores vs predictores

Esta distinción es básica y conviene interiorizarla pronto.

### Transformador
Recibe datos y devuelve datos transformados.

Ejemplos:
- `StandardScaler`
- `SimpleImputer`
- `PCA`

### Predictor
Recibe datos y aprende a predecir la salida `y`.

Ejemplos:
- `LogisticRegression`
- `KNeighborsClassifier`
- `DecisionTreeClassifier`
- `RandomForestClassifier`

¿Por qué importa esta diferencia?

Porque en un flujo real de ML casi nunca trabajamos con los datos "en bruto".  
Muy a menudo necesitamos:

1. transformar los datos,
2. aplicar la misma transformación a nuevos datos,
3. y solo después predecir.

Por eso `Pipeline` es una herramienta tan importante.

## 7. Módulos típicos de `scikit-learn`

No hace falta memorizarlos todos de golpe, pero sí tener un mapa mental:

- `sklearn.datasets`
- `sklearn.model_selection`
- `sklearn.preprocessing`
- `sklearn.pipeline`
- `sklearn.metrics`
- `sklearn.dummy`
- `sklearn.linear_model`
- `sklearn.neighbors`
- `sklearn.tree`
- `sklearn.ensemble`

Con solo estas familias ya se puede hacer muchísimo trabajo en tabular.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 8. Carga del dataset

Vamos a cargar el dataset con `load_breast_cancer`.  
Además, si no existe una copia local en CSV, la guardaremos para que el notebook sea cómodo de reutilizar.

In [ ]:
csv_path = Path("breast_cancer_classification.csv")

if csv_path.exists():
    df = pd.read_csv(csv_path)
else:
    data_bunch = load_breast_cancer(as_frame=True)
    df = data_bunch.frame.copy()
    df["diagnosis"] = df["target"].map({0: "malignant", 1: "benign"})
    df = df.drop(columns="target")
    df.to_csv(csv_path, index=False)

df.head()

## 9. Primera inspección del dataset

Antes de modelar, conviene responder preguntas muy básicas:

- ¿cuántas filas y columnas hay?
- ¿qué tipos de variables tenemos?
- ¿cuál será la variable objetivo?
- ¿hay valores ausentes?
- ¿cómo se reparte la clase?

In [ ]:
print(f"Dimensiones del dataset: {df.shape}")
print("\nTipos de dato:")
display(df.dtypes.value_counts())

In [ ]:
df.info()

## 10. La variable objetivo

La columna que queremos predecir es `diagnosis`.

Las dos clases posibles son:

- `malignant`
- `benign`

Este detalle es importante para las métricas: en muchos problemas no todos los errores son igual de graves desde el punto de vista del dominio.

In [ ]:
df["diagnosis"].value_counts()

In [ ]:
class_counts = df["diagnosis"].value_counts()

plt.figure(figsize=(6, 4))
class_counts.plot(kind="bar")
plt.title("Distribución de la variable objetivo")
plt.xlabel("Clase")
plt.ylabel("Número de observaciones")
plt.xticks(rotation=0)
plt.show()

## 11. Comentario sobre el balance de clases

Cuando una clase es mucho más frecuente que otra, la accuracy puede resultar engañosa.

Ejemplo mental:
si el 95% de los casos fueran `benign`, un modelo muy malo podría conseguir 95% de accuracy diciendo siempre `benign`.

Por eso, en clasificación, casi siempre debemos mirar algo más que accuracy.

## 12. Definimos `X` e `y`

Este paso es muy importante:

- `X`: variables de entrada,
- `y`: variable objetivo.

In [ ]:
X = df.drop(columns="diagnosis")
y = df["diagnosis"]

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

## 13. Pequeña exploración visual

No vamos a hacer aquí un EDA exhaustivo.  
Solo queremos ver que:

- hay variables con escalas distintas,
- algunas parecen separar mejor las clases que otras,
- y el problema parece tener cierta estructura.

In [ ]:
selected_features = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, feature in zip(axes, selected_features):
    sns.histplot(data=df, x=feature, hue="diagnosis", kde=True, ax=ax, element="step")
    ax.set_title(feature)

plt.tight_layout()
plt.show()

## 14. `train_test_split`: por qué separamos los datos

No queremos evaluar el modelo con los mismos datos con los que lo entrenamos.  
Si lo hiciéramos, correríamos el riesgo de medir memoria en lugar de generalización.

En clasificación suele ser muy buena idea usar `stratify=y`, para conservar aproximadamente la misma proporción de clases en train y test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("\nDistribución en train:")
display(y_train.value_counts(normalize=True).rename("proportion"))
print("\nDistribución en test:")
display(y_test.value_counts(normalize=True).rename("proportion"))

## 15. Un transformador muy importante: `StandardScaler`

Muchas variables numéricas están en escalas muy distintas.

Eso importa especialmente en modelos como:

- KNN,
- regresión logística,
- y otros modelos sensibles a magnitudes y distancias.

`StandardScaler` es un **transformador** porque:

- aprende estadísticas en `fit`,
- y luego transforma datos con `transform`.

In [ ]:
scaler_demo = StandardScaler()

X_train_scaled_demo = scaler_demo.fit_transform(X_train)
X_test_scaled_demo = scaler_demo.transform(X_test)

print("Media aprendida para las primeras 5 variables:")
print(np.round(scaler_demo.mean_[:5], 3))

print("\nDesviación típica aprendida para las primeras 5 variables:")
print(np.round(scaler_demo.scale_[:5], 3))

print("\nForma de X_train escalado:", X_train_scaled_demo.shape)
print("Forma de X_test escalado :", X_test_scaled_demo.shape)

## 16. Muy importante: evitar *data leakage*

Nunca debemos ajustar un transformador usando información del conjunto de test.

Mal:
- escalar todo el dataset y luego dividir.

Bien:
1. dividir train y test,
2. ajustar el transformador con train,
3. aplicar la transformación a train y a test.

La forma más cómoda y segura de automatizar esto es `Pipeline`.

## 17. Primer baseline: `DummyClassifier`

Antes de usar modelos sofisticados, conviene construir una referencia muy simple.

`DummyClassifier(strategy="most_frequent")` predice siempre la clase mayoritaria.

Si un modelo complejo no mejora claramente sobre esto, hay un problema.

In [ ]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)

y_pred_dummy = dummy_clf.predict(X_test)

print("Accuracy del baseline dummy:", round(accuracy_score(y_test, y_pred_dummy), 4))
print("\nDistribución de predicciones del dummy:")
display(pd.Series(y_pred_dummy).value_counts())

## 18. Primer clasificador serio: `LogisticRegression`

Aunque el nombre contenga la palabra "regresión", `LogisticRegression` es un **clasificador**.

Es un gran punto de partida porque:

- suele funcionar bien como baseline serio,
- es rápido,
- permite obtener probabilidades,
- y es relativamente interpretable.

Como nuestras variables están en escalas distintas, la usaremos dentro de un `Pipeline` con `StandardScaler`.

In [ ]:
logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])

logreg_pipe.fit(X_train, y_train)

y_pred_logreg = logreg_pipe.predict(X_test)

print("Accuracy en test:", round(accuracy_score(y_test, y_pred_logreg), 4))

## 19. `Pipeline`: por qué merece la pena acostumbrarse a él

`Pipeline` encadena pasos en orden.

En nuestro caso:

1. el escalador aprende con `X_train`,
2. transforma los datos,
3. y el modelo aprende con esos datos transformados.

Luego, al predecir:

1. se transforma `X_test` con lo aprendido en train,
2. y después se genera la predicción.

Ventajas:

- evita fugas de información con más facilidad,
- hace el código más limpio,
- se integra muy bien con validación cruzada,
- se integra muy bien con `GridSearchCV`.

## 20. Primer vistazo a métricas de clasificación

Empezamos con la más famosa:

### Accuracy
Porcentaje de ejemplos clasificados correctamente.

Pero, como veremos, no conviene quedarse solo con ella.

In [ ]:
print(classification_report(y_test, y_pred_logreg))

## 21. Cómo leer un `classification_report`

Para cada clase, el informe nos muestra:

### Precision
De todo lo que el modelo predijo como esa clase, ¿qué proporción era realmente de esa clase?

### Recall
De todos los casos reales de esa clase, ¿qué proporción ha detectado?

### F1-score
Equilibrio entre precision y recall.

### Support
Número de ejemplos reales de esa clase en el conjunto evaluado.

Este tipo de tabla ya nos da una visión mucho más rica que un único número.

## 22. Matriz de confusión

La matriz de confusión permite ver con mucha claridad qué errores comete el modelo.

Nos ayuda a distinguir entre:

- aciertos sobre positivos,
- aciertos sobre negativos,
- falsos positivos,
- falsos negativos.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_logreg,
    cmap="Blues",
    ax=ax
)
plt.title("Matriz de confusión — Logistic Regression")
plt.show()

## 23. `predict` frente a `predict_proba`

Ya hemos usado `predict`, que devuelve la clase final.

Ahora inspeccionamos `predict_proba`, que nos devuelve probabilidades por clase.

Eso nos permitirá:

- ver la confianza del modelo,
- calcular ROC-AUC,
- y experimentar con distintos umbrales.

In [ ]:
proba_logreg = logreg_pipe.predict_proba(X_test)

print("Forma de la matriz de probabilidades:", proba_logreg.shape)
print("Orden de las clases:", logreg_pipe.named_steps["model"].classes_)
print("\nPrimeras 5 filas de predict_proba:")
print(np.round(proba_logreg[:5], 4))

## 24. Probabilidad de la clase `malignant`

Como `predict_proba` devuelve una columna por clase, localizamos la columna correspondiente a `malignant`.

In [ ]:
classes_ = logreg_pipe.named_steps["model"].classes_
malignant_idx = list(classes_).index("malignant")

y_proba_malignant = proba_logreg[:, malignant_idx]

print("Índice de malignant:", malignant_idx)
print("Primeras probabilidades de malignant:")
print(np.round(y_proba_malignant[:10], 4))

## 25. Umbral de decisión

Por defecto, el modelo convierte probabilidades en clases con un umbral implícito cercano a 0.5.

Pero ese umbral se puede mover.

- Si bajamos el umbral para declarar `malignant`, detectaremos más casos malignos.
- Pero probablemente aumentarán los falsos positivos.

Esto introduce uno de los grandes compromisos de la clasificación aplicada.

In [ ]:
def predict_with_threshold(proba_malignant, threshold=0.5):
    return np.where(proba_malignant >= threshold, "malignant", "benign")

for threshold in [0.50, 0.35, 0.20]:
    y_pred_thr = predict_with_threshold(y_proba_malignant, threshold=threshold)
    print(f"\nUmbral = {threshold}")
    print("Precision (malignant):", round(precision_score(y_test, y_pred_thr, pos_label="malignant"), 4))
    print("Recall    (malignant):", round(recall_score(y_test, y_pred_thr, pos_label="malignant"), 4))
    print("F1        (malignant):", round(f1_score(y_test, y_pred_thr, pos_label="malignant"), 4))

## 26. Qué queremos que quede claro del umbral

No hace falta memorizar qué valor de umbral ha salido mejor aquí.

Lo importante es entender la lógica:

- cuando somos más permisivos con la etiqueta positiva, suele subir el recall,
- cuando somos más exigentes, suele subir la precision,
- y rara vez podemos maximizar ambas cosas simultáneamente.

# Parte B. Evaluación, validación e hiperparámetros

## 27. ¿Por qué accuracy no basta?

Dos modelos pueden tener accuracies parecidas y, sin embargo, comportarse de forma muy distinta sobre la clase que más nos importa.

Por eso, en clasificación, debemos pensar en:

- qué errores comete el modelo,
- cuántos,
- y sobre qué clase.

## 28. Métricas que conviene conocer

### Accuracy
Proporción total de aciertos.

### Precision
Cuando el modelo dice "positivo", ¿cuántas veces acierta?

### Recall
De todos los positivos reales, ¿cuántos detecta?

### F1
Compromiso entre precision y recall.

### ROC-AUC
Capacidad de separar clases a distintos umbrales.

No hay una métrica universalmente perfecta.  
La métrica correcta depende del problema.

## 29. Validación cruzada: la idea

Una sola partición train/test puede darnos una estimación algo dependiente del azar de esa partición.

La validación cruzada intenta reducir ese problema:

1. divide el conjunto de entrenamiento en varios pliegues,
2. entrena en una parte,
3. valida en la otra,
4. repite rotando,
5. y resume el rendimiento.

## 30. Muy importante: el test se reserva para el final

Una forma sana de trabajar es:

- `train`: ajustar,
- validación cruzada sobre `train`: comparar y seleccionar,
- `test`: evaluación final honesta.

Si toqueteamos demasiado el test, deja de ser realmente final.

## 31. Construimos varios modelos para comparar

Vamos a comparar varias familias muy típicas en problemas tabulares:

- `LogisticRegression`
- `KNeighborsClassifier`
- `DecisionTreeClassifier`
- `RandomForestClassifier`

Observa que algunos modelos llevan escalado y otros no.

### Modelos que suelen beneficiarse del escalado
- logística,
- KNN.

### Modelos basados en árboles
- árbol de decisión,
- random forest.

Estos últimos no dependen tanto de la escala de las variables.

In [ ]:
models = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),
    "DecisionTree": Pipeline([
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "RandomForest": Pipeline([
        ("model", RandomForestClassifier(random_state=RANDOM_STATE))
    ]),
}

## 32. Definimos métricas explícitas

En vez de depender del `.score()` por defecto, vamos a declarar de forma explícita las métricas.

Tomaremos `malignant` como clase positiva de interés.

In [ ]:
f1_malignant_scorer = make_scorer(f1_score, pos_label="malignant")

scoring = {
    "accuracy": "accuracy",
    "precision_malignant": make_scorer(precision_score, pos_label="malignant"),
    "recall_malignant": make_scorer(recall_score, pos_label="malignant"),
    "f1_malignant": f1_malignant_scorer,
    "roc_auc": "roc_auc",
}

## 33. Ejecutamos `cross_validate`

Fíjate en que la validación cruzada se ejecuta **sobre `X_train` e `y_train`**, no sobre test.

In [ ]:
cv_rows = []

for name, model in models.items():
    cv_result = cross_validate(
        model,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    row = {"model": name}
    for metric_name, values in cv_result.items():
        if metric_name.startswith("test_"):
            clean_name = metric_name.replace("test_", "")
            row[f"{clean_name}_mean"] = np.mean(values)
            row[f"{clean_name}_std"] = np.std(values)
    cv_rows.append(row)

cv_results_df = pd.DataFrame(cv_rows).sort_values(by="f1_malignant_mean", ascending=False)
cv_results_df

## 34. Cómo leer la tabla de validación cruzada

Ahora no solo vemos "quién gana", sino también:

- la media de cada métrica,
- la variación entre pliegues,
- y el equilibrio entre precisión y recall sobre la clase de interés.

In [ ]:
metric_cols = [
    "accuracy_mean",
    "precision_malignant_mean",
    "recall_malignant_mean",
    "f1_malignant_mean",
    "roc_auc_mean",
]

plot_df = cv_results_df.set_index("model")[metric_cols]

plot_df.plot(kind="bar", figsize=(12, 5))
plt.title("Comparación de modelos mediante validación cruzada")
plt.ylabel("Score medio CV")
plt.ylim(0.85, 1.01)
plt.xticks(rotation=0)
plt.show()

## 35. Primera conclusión comparativa

En tabular es muy habitual que varios modelos se comporten bien:

- lineales,
- basados en distancias,
- árboles,
- ensembles.

Una gran lección de ML aplicado es esta:  
**no existe un modelo universalmente mejor**.

## 36. Elegimos un candidato para ajustar hiperparámetros: KNN

`KNeighborsClassifier` es muy pedagógico porque sus hiperparámetros tienen una interpretación bastante intuitiva:

- `n_neighbors`: cuántos vecinos consultamos,
- `weights`: si todos los vecinos pesan igual o si los cercanos pesan más,
- `p`: tipo de distancia.

In [ ]:
knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

param_grid_knn = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

grid_knn = GridSearchCV(
    estimator=knn_pipe,
    param_grid=param_grid_knn,
    scoring=f1_malignant_scorer,
    cv=5,
    n_jobs=-1,
    refit=True,
)

grid_knn.fit(X_train, y_train)

## 37. Qué hace exactamente `GridSearchCV`

`GridSearchCV`:

1. prueba todas las combinaciones del grid,
2. evalúa cada combinación con validación cruzada,
3. se queda con la mejor según la métrica elegida,
4. y reentrena automáticamente ese mejor modelo sobre todo el conjunto de entrenamiento.

In [ ]:
print("Mejores hiperparámetros KNN:")
print(grid_knn.best_params_)

print("\nMejor score medio CV:")
print(round(grid_knn.best_score_, 4))

In [ ]:
knn_cv_results = pd.DataFrame(grid_knn.cv_results_)
cols_to_show = [
    "param_model__n_neighbors",
    "param_model__weights",
    "param_model__p",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]
knn_cv_results[cols_to_show].sort_values(by="rank_test_score").head(10)

## 38. Segundo ejemplo de ajuste: `RandomForestClassifier`

No queremos solo un ejemplo.  
También es útil ver que otra familia de modelos tiene otros hiperparámetros y otra lógica.

En Random Forest nos fijaremos en:

- `n_estimators`,
- `max_depth`,
- `min_samples_leaf`.

In [ ]:
rf_pipe = Pipeline([
    ("model", RandomForestClassifier(random_state=RANDOM_STATE))
])

param_grid_rf = {
    "model__n_estimators": [100],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_leaf": [1, 2],
}

grid_rf = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid_rf,
    scoring=f1_malignant_scorer,
    cv=5,
    n_jobs=-1,
    refit=True,
)

grid_rf.fit(X_train, y_train)

In [ ]:
print("Mejores hiperparámetros Random Forest:")
print(grid_rf.best_params_)

print("\nMejor score medio CV:")
print(round(grid_rf.best_score_, 4))

## 39. Comparación final sobre test

Ahora sí pasamos al conjunto de test para una evaluación final honesta.

Vamos a comparar:

- baseline dummy,
- regresión logística,
- mejor KNN,
- mejor Random Forest.

In [ ]:
final_models = {
    "Dummy": dummy_clf,
    "LogisticRegression": logreg_pipe,
    "BestKNN": grid_knn.best_estimator_,
    "BestRandomForest": grid_rf.best_estimator_,
}

test_rows = []

for name, model in final_models.items():
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        classes = model.named_steps["model"].classes_ if hasattr(model, "named_steps") else model.classes_
        malignant_idx = list(classes).index("malignant")
        y_proba = model.predict_proba(X_test)[:, malignant_idx]
        roc_auc = roc_auc_score((y_test == "malignant").astype(int), y_proba)
    else:
        roc_auc = np.nan

    test_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_malignant": precision_score(y_test, y_pred, pos_label="malignant"),
        "recall_malignant": recall_score(y_test, y_pred, pos_label="malignant"),
        "f1_malignant": f1_score(y_test, y_pred, pos_label="malignant"),
        "roc_auc": roc_auc,
    })

test_results_df = pd.DataFrame(test_rows).sort_values(by="f1_malignant", ascending=False)
test_results_df

## 40. Cómo pensar la tabla final

No se trata solo de mirar qué fila tiene el número más grande.

Conviene preguntarse:

- ¿qué mejora real hay respecto al baseline?
- ¿qué modelo equilibra mejor precision y recall?
- ¿las diferencias son grandes o pequeñas?
- ¿merece la pena la complejidad extra?

## 41. Analizamos a fondo el mejor modelo según F1 en test

Tomamos el modelo con mejor `f1_malignant` y lo inspeccionamos con más detalle.

In [ ]:
best_model_name = test_results_df.iloc[0]["model"]
best_model = final_models[best_model_name]

print("Modelo elegido para análisis detallado:", best_model_name)

y_pred_best = best_model.predict(X_test)
print("\nClassification report:")
print(classification_report(y_test, y_pred_best))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_best,
    cmap="Purples",
    ax=ax
)
plt.title(f"Matriz de confusión — {best_model_name}")
plt.show()

## 42. Curva ROC y ROC-AUC

La curva ROC resume cómo cambia la relación entre verdaderos positivos y falsos positivos cuando movemos el umbral.

La medida ROC-AUC se interpreta, de forma intuitiva, como la capacidad del modelo para separar positivos y negativos a lo largo de muchos posibles umbrales.

In [ ]:
plt.figure(figsize=(7, 6))

for name, model in final_models.items():
    if hasattr(model, "predict_proba"):
        classes = model.named_steps["model"].classes_ if hasattr(model, "named_steps") else model.classes_
        malignant_idx = list(classes).index("malignant")
        y_score = model.predict_proba(X_test)[:, malignant_idx]

        RocCurveDisplay.from_predictions(
            (y_test == "malignant").astype(int),
            y_score,
            name=name,
            ax=plt.gca()
        )

plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("Curvas ROC en el conjunto de test")
plt.show()

## 43. Inspección manual de algunos casos

A veces ayuda mucho mirar unos pocos ejemplos concretos con:

- clase real,
- clase predicha,
- probabilidad asignada a `malignant`.

In [ ]:
if hasattr(best_model, "predict_proba"):
    classes = best_model.named_steps["model"].classes_ if hasattr(best_model, "named_steps") else best_model.classes_
    malignant_idx = list(classes).index("malignant")
    proba_best = best_model.predict_proba(X_test)[:, malignant_idx]
else:
    proba_best = np.full(shape=len(X_test), fill_value=np.nan)

inspection_df = X_test.copy()
inspection_df["true_label"] = y_test.values
inspection_df["pred_label"] = y_pred_best
inspection_df["proba_malignant"] = proba_best

inspection_df.head(10)

## 44. Un vistazo a la interpretabilidad: importancia de variables en Random Forest

Los modelos basados en árboles permiten extraer importancias de variables.

Hay que interpretarlas con prudencia:

- no son causalidad,
- dependen del modelo,
- y no reemplazan un análisis profundo del dominio.

In [ ]:
best_rf = grid_rf.best_estimator_
rf_model = best_rf.named_steps["model"]

feature_importances = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feature_importances.head(12)

In [ ]:
plt.figure(figsize=(8, 6))
feature_importances.head(12).sort_values().plot(kind="barh")
plt.title("Variables más importantes según Random Forest")
plt.xlabel("Importancia")
plt.show()

## 45. ¿Cuándo elegirías cada familia de modelos?

### Regresión logística
Muy buena como baseline serio, rápida y bastante interpretable.

### KNN
Muy intuitivo, pero sensible a la escala y al ruido.

### Árbol de decisión
Fácil de explicar, pero con riesgo de sobreajuste.

### Random Forest
Suele rendir muy bien en tabular y es bastante robusto.

## 46. Parámetros frente a hiperparámetros

### Parámetros
Los aprende el modelo durante `fit`.

### Hiperparámetros
Los decide la persona que diseña el experimento antes del entrenamiento.

`GridSearchCV` busca hiperparámetros, no parámetros.

## 47. Errores frecuentes en clasificación con `scikit-learn`

1. Mirar solo accuracy.  
2. Escalar antes de separar train y test.  
3. No usar `stratify` cuando conviene.  
4. Tocar demasiadas veces el test.  
5. Comparar modelos con preprocesados inconsistentes.  
6. Elegir el modelo "más famoso" en vez del más adecuado.

## 48. Resumen del flujo correcto

1. Entender el problema y elegir métrica.
2. Separar train y test.
3. Crear baseline.
4. Construir pipelines correctos.
5. Comparar con validación cruzada sobre train.
6. Ajustar hiperparámetros.
7. Elegir modelo.
8. Evaluar una vez, de forma final, sobre test.

## 49. Mini ejercicio guiado 1

Intenta responder con tus palabras:

1. ¿Por qué `StandardScaler` es un transformador?
2. ¿Por qué `Pipeline` ayuda a evitar fugas de información?
3. ¿Para qué sirve un `DummyClassifier`?
4. ¿Por qué accuracy puede ser insuficiente?

## 50. Mini ejercicio guiado 2

Piensa qué esperas que pase si:

1. quitamos el escalado antes de KNN,
2. usamos muy pocos vecinos,
3. usamos demasiados vecinos,
4. bajamos mucho el umbral para detectar `malignant`.

## 51. Mini ejercicio guiado 3

Prueba por tu cuenta a modificar:

- el `random_state`,
- el `test_size`,
- la métrica del `GridSearchCV`,
- o el grid de hiperparámetros.

## 52. Conclusiones de la sesión

En esta sesión hemos visto que:

- clasificación no es simplemente "regresión con etiquetas",
- `scikit-learn` organiza muy bien el flujo mediante una API coherente,
- distinguir transformadores y predictores es clave,
- `Pipeline` no es decoración: ayuda a trabajar correctamente,
- accuracy es útil, pero no siempre suficiente,
- precision, recall, F1 y ROC-AUC permiten una evaluación mucho más rica,
- validación cruzada y `GridSearchCV` ayudan a comparar y ajustar con rigor,
- y el conjunto de test debe reservarse para la evaluación final honesta.

Dicho de forma breve:

**clasificar bien no es solo entrenar un algoritmo; es diseñar bien todo el experimento.**

## 53. Posibles ampliaciones

Si quisieras seguir profundizando, algunas ampliaciones muy naturales serían:

1. repetir el flujo con un dataset multiclase,
2. añadir `PCA`,
3. estudiar calibración de probabilidades,
4. ver problemas con desbalanceo más severo,
5. introducir `class_weight`.

## 54. Celda libre para experimentos del alumnado

In [ ]:
# Escribe aquí tus pruebas y experimentos.